In [2]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.model_selection import train_test_split

import src.features as ft


train_df, test_df= (pd.read_csv('../data/raw/train.csv'), pd.read_csv('../data/raw/test.csv'))

RANDOMSTATE = 42

train_fe = ft.create_features(train_df)
test_fe = ft.create_features(test_df)

In [3]:
from src.pipeline import (
    numeric_features,
    categorical_features,
    binary_features

)

X = train_fe[numeric_features + categorical_features + binary_features]
y = train_fe['Survived']


X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOMSTATE,
    stratify=y
)



In [4]:
display(X_train.head())
display(X_val.head())

,Age,Fare,SibSp,Parch,FamilySize,Pclass,Sex,Embarked,Title,AgeWasMissing,CabinWasMissing,IsAlone
692,NaN,56.4958,0,0,1,3,male,S,Mr,1,1,1
481,NaN,0.0000,0,0,1,2,male,S,Mr,1,1,1
527,NaN,221.7792,0,0,1,1,male,S,Mr,1,0,1
855,18.0,9.3500,0,1,2,3,female,S,Mrs,0,1,0
801,31.0,26.2500,1,1,3,2,female,S,Mrs,0,1,0


,Age,Fare,SibSp,Parch,FamilySize,Pclass,Sex,Embarked,Title,AgeWasMissing,CabinWasMissing,IsAlone
565,24.0,24.1500,2,0,3,3,male,S,Mr,0,1,0
160,44.0,16.1000,0,1,2,3,male,S,Mr,0,1,0
553,22.0,7.2250,0,0,1,3,male,C,Mr,0,1,1
860,41.0,14.1083,2,0,3,3,male,S,Mr,0,1,0
241,NaN,15.5000,1,0,2,3,female,Q,Miss,1,1,0


In [5]:
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import cross_validate
from src.pipeline import make_preprocessor


preprocessor = make_preprocessor(scale=True)


models = {
    "knn": KNeighborsClassifier(n_neighbors=5),

    "lr": LogisticRegression(
        max_iter=5000
    ),

    "dt": DecisionTreeClassifier(
        random_state=RANDOMSTATE
    ),

    "rf": RandomForestClassifier(
        random_state=RANDOMSTATE
    ),

    "extra_trees": ExtraTreesClassifier(
        random_state=RANDOMSTATE
    ),

    "gradient_boosting": GradientBoostingClassifier(
        random_state=RANDOMSTATE
    ),

    "adaboost": AdaBoostClassifier(
        random_state=RANDOMSTATE
    ),
}

results = []

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}


for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", clone(model)),
    ])

    # 1. Validation 성능
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_val)

    val_accuracy = accuracy_score(y_val, y_pred)
    val_precision = precision_score(
        y_val,
        y_pred,
        zero_division=0
    )
    val_recall = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )
    val_f1 = f1_score(
        y_val,
        y_pred,
        zero_division=0
    )


    # 2. Cross Validation
    cv_result = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        n_jobs=-1,
    )


    results.append({
        "model": name,

        "val_accuracy": val_accuracy,
        "val_precision": val_precision,
        "val_recall": val_recall,
        "val_f1": val_f1,

        "cv_accuracy": cv_result[
            "test_accuracy"
        ].mean(),

        "cv_precision": cv_result[
            "test_precision"
        ].mean(),

        "cv_recall": cv_result[
            "test_recall"
        ].mean(),

        "cv_f1": cv_result[
            "test_f1"
        ].mean(),
    })


result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    "cv_f1",
    ascending=False
)

display(
    result_df.round(4)
)

,model,val_accuracy,val_precision,val_recall,val_f1,cv_accuracy,cv_precision,cv_recall,cv_f1
1,lr,0.8380,0.8030,0.7681,0.7852,0.8203,0.7810,0.7399,0.7591
6,adaboost,0.8156,0.7500,0.7826,0.7660,0.8147,0.7562,0.7622,0.7587
5,gradient_boosting,0.7989,0.7895,0.6522,0.7143,0.8175,0.7964,0.7035,0.7459
0,knn,0.8212,0.7937,0.7246,0.7576,0.8119,0.7831,0.7036,0.7394
3,rf,0.8156,0.7571,0.7681,0.7626,0.7909,0.7353,0.7145,0.7244
4,extra_trees,0.7821,0.7083,0.7391,0.7234,0.7783,0.7114,0.7147,0.7126
2,dt,0.7765,0.7164,0.6957,0.7059,0.7501,0.6635,0.7036,0.6827


In [6]:
import plotly.express as px

metrics = [
    "cv_accuracy",
    "cv_precision",
    "cv_recall",
    "cv_f1",
]

plot_df = result_df.melt(
    id_vars="model",
    value_vars=metrics,
    var_name="metric",
    value_name="score",
)

fig = px.scatter(
    result_df,
    x="cv_precision",
    y="cv_recall",
    text="model",
    size="cv_f1",
    title="Precision vs Recall by Model"
)

fig.update_traces(
    textposition="top center"
)

fig.show()

In [35]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV


def run_grid_search(
    model_configs,
    param_grids,
    X_train,
    y_train,
    scoring,
    cv
):
    grid_results = {}
    best_models = {}

    for name, config in model_configs.items():

        model = config["model"]
        scale = config["scale"]

        pipeline = Pipeline([
            (
                "preprocessor",
                make_preprocessor(scale=scale)
            ),
            ("model", model),
        ])

        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grids[name],
            cv=cv,
            scoring=scoring,
            refit="f1",
            n_jobs=-1,
        )

        grid.fit(X_train, y_train)

        best_idx = grid.best_index_

        grid_results[name] = {
            "best_params": grid.best_params_,
            "cv_accuracy": grid.cv_results_[
                "mean_test_accuracy"
            ][best_idx],
            "cv_precision": grid.cv_results_[
                "mean_test_precision"
            ][best_idx],
            "cv_recall": grid.cv_results_[
                "mean_test_recall"
            ][best_idx],
            "cv_f1": grid.cv_results_[
                "mean_test_f1"
            ][best_idx],
        }

        best_models[name] = grid.best_estimator_

    return grid_results, best_models



def run_random_search(
    model_configs,
    param_distributions,
    X_train,
    y_train,
    scoring,
    n_iter=30,
):
    random_results = {}
    best_models = {}
    random_searches = {}

    for name, config in model_configs.items():

        model = config["model"]
        scale = config["scale"]

        pipeline = Pipeline([
            ("preprocessor", make_preprocessor(scale=scale)),
            ("model", model),
        ])

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_distributions[name],
            n_iter=n_iter,
            cv=cv,
            scoring=scoring,
            refit="f1",
            n_jobs=-1,
            random_state=RANDOMSTATE,
        )

        search.fit(X_train, y_train)

        best_idx = search.best_index_

        random_results[name] = {
            "best_params": search.best_params_,
            "cv_accuracy": search.cv_results_[
                "mean_test_accuracy"
            ][best_idx],
            "cv_precision": search.cv_results_[
                "mean_test_precision"
            ][best_idx],
            "cv_recall": search.cv_results_[
                "mean_test_recall"
            ][best_idx],
            "cv_f1": search.cv_results_[
                "mean_test_f1"
            ][best_idx],
        }

        best_models[name] = search.best_estimator_

        # RandomizedSearchCV 객체 자체 저장
        random_searches[name] = search

    return random_results, best_models, random_searches

def run_random_search(
    model_configs,
    param_distributions,
    X_train,
    y_train,
    scoring,
    cv,
    n_iter=30,
):
    summary = {}
    best_models = {}
    searches = {}
    trials = {}

    for name, config in model_configs.items():

        pipeline = Pipeline([
            (
                "preprocessor",
                make_preprocessor(scale=config["scale"])
            ),
            ("model", config["model"]),
        ])

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_distributions[name],
            n_iter=n_iter,
            cv=cv,
            scoring=scoring,
            refit="f1",
            n_jobs=-1,
            random_state=RANDOMSTATE,
        )

        search.fit(X_train, y_train)

        # ---------------------------------
        # 모든 Random Search 결과
        # ---------------------------------
        cv_df = pd.DataFrame(search.cv_results_)

        result_df = (
            cv_df[
                [
                    "rank_test_f1",
                    "mean_test_f1",
                    "std_test_f1",
                    "mean_test_accuracy",
                    "mean_test_precision",
                    "mean_test_recall",
                    "params",
                ]
            ]
            .sort_values("rank_test_f1")
            .reset_index(drop=True)
        )

        # 어떤 모델의 결과인지 표시
        result_df.insert(0, "model", name)

        # ---------------------------------
        # Best 결과 요약
        # ---------------------------------
        best_idx = search.best_index_

        summary[name] = {
            "best_params": search.best_params_,
            "cv_accuracy": cv_df.loc[
                best_idx, "mean_test_accuracy"
            ],
            "cv_precision": cv_df.loc[
                best_idx, "mean_test_precision"
            ],
            "cv_recall": cv_df.loc[
                best_idx, "mean_test_recall"
            ],
            "cv_f1": cv_df.loc[
                best_idx, "mean_test_f1"
            ],
            "cv_f1_std": cv_df.loc[
                best_idx, "std_test_f1"
            ],
        }

        # 최적 모델
        best_models[name] = search.best_estimator_

        # Search 객체 자체
        searches[name] = search

        # 모든 trial 결과
        trials[name] = result_df

    summary_df = (
        pd.DataFrame(summary)
        .T
        .sort_values("cv_f1", ascending=False)
    )

    return summary_df, best_models, searches, trials

In [8]:
model_configs = {
    "knn": {
        "model": KNeighborsClassifier(),
        "scale": True,
    },

    "lr": {
        "model": LogisticRegression(max_iter=5000),
        "scale": True,
    },

    "rf": {
        "model": RandomForestClassifier(
            random_state=RANDOMSTATE
        ),
        "scale": False,
    },

    "gradient_boosting": {
        "model": GradientBoostingClassifier(
            random_state=RANDOMSTATE
        ),
        "scale": False,
    },

    "adaboost": {
        "model": AdaBoostClassifier(
            random_state=RANDOMSTATE
        ),
        "scale": False,
    },
}

In [9]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

In [10]:
from scipy.stats import randint, uniform, loguniform
from sklearn.model_selection import StratifiedKFold

param_distributions = {
    "knn": {
        "model__n_neighbors": randint(3, 31),
        "model__weights": ["uniform", "distance"],
        "model__metric": ["euclidean", "manhattan"],
    },

    "lr": {
        "model__C": loguniform(1e-3, 1e2),
        "model__penalty": ["l1", "l2"],
        "model__solver": ["liblinear"],
        "model__class_weight": [None, "balanced"],
    },

    "rf": {
        "model__n_estimators": randint(100, 801),
        "model__max_depth": [None, 5, 10, 15, 20, 30],
        "model__min_samples_split": randint(2, 21),
        "model__min_samples_leaf": randint(1, 11),
        "model__max_features": ["sqrt", "log2", None],
        "model__criterion": ["gini", "entropy"],
        "model__class_weight": [None, "balanced"],
    },

    "gradient_boosting": {
        "model__n_estimators": randint(50, 401),
        "model__learning_rate": loguniform(0.01, 0.3),
        "model__max_depth": randint(1, 6),
        "model__min_samples_split": randint(2, 21),
        "model__min_samples_leaf": randint(1, 11),
        "model__subsample": uniform(0.5, 0.5),
    },

    "adaboost": {
        "model__n_estimators": randint(50, 501),
        "model__learning_rate": loguniform(0.01, 1.0),
    },
}

cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOMSTATE
    )

random_summary, random_best_models, random_searches, random_trials = run_random_search(
    model_configs=model_configs,
    param_distributions=param_distributions,
    X_train=X_train,
    y_train=y_train,
    scoring=scoring,
    n_iter=30,
    cv=cv
)

/Users/yundong-ug/dev/boot_camp/kaggle-titanic-ml/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/yundong-ug/dev/boot_camp/kaggle-titanic-ml/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/yundong-ug/dev/boot_camp/kaggle-titanic-ml/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, l

In [11]:
display(random_summary)

,best_params,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_f1_std
rf,"{'model__class_weight': 'balanced', 'model__cr...",0.837152,0.784553,0.802222,0.791587,0.047904
gradient_boosting,"{'model__learning_rate': 0.13766134492174428, ...",0.842736,0.827615,0.750976,0.786068,0.024957
adaboost,"{'model__learning_rate': 0.46225890010208287, ...",0.827253,0.775955,0.776566,0.775455,0.025421
lr,"{'model__C': 2.6373339933815254, 'model__class...",0.828661,0.792473,0.758316,0.773422,0.027021
knn,"{'model__metric': 'manhattan', 'model__n_neigh...",0.82024,0.811999,0.695758,0.747685,0.029837


In [12]:
display(
    random_trials["rf"].head(10)
)

,model,rank_test_f1,mean_test_f1,std_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,params
0,rf,1,0.791587,0.047904,0.837152,0.784553,0.802222,"{'model__class_weight': 'balanced', 'model__cr..."
1,rf,2,0.789234,0.048287,0.835743,0.783819,0.798519,"{'model__class_weight': 'balanced', 'model__cr..."
2,rf,3,0.787050,0.041538,0.831537,0.772548,0.805926,"{'model__class_weight': 'balanced', 'model__cr..."
3,rf,4,0.783913,0.036506,0.834325,0.786148,0.783906,"{'model__class_weight': 'balanced', 'model__cr..."
4,rf,5,0.783418,0.034968,0.832926,0.787797,0.783973,"{'model__class_weight': 'balanced', 'model__cr..."
5,rf,6,0.782503,0.045482,0.831518,0.780940,0.787609,"{'model__class_weight': 'balanced', 'model__cr..."
6,rf,7,0.778321,0.035786,0.831547,0.790906,0.769428,"{'model__class_weight': 'balanced', 'model__cr..."
7,rf,8,0.773000,0.021865,0.832887,0.811162,0.739865,"{'model__class_weight': None, 'model__criterio..."
8,rf,9,0.772634,0.026535,0.824485,0.772184,0.776700,"{'model__class_weight': 'balanced', 'model__cr..."
9,rf,10,0.771136,0.047717,0.818891,0.755362,0.791178,"{'model__class_weight': 'balanced', 'model__cr..."


In [13]:
display(
    random_trials["gradient_boosting"].head(10)
)

,model,rank_test_f1,mean_test_f1,std_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,params
0,gradient_boosting,1,0.786068,0.024957,0.842736,0.827615,0.750976,"{'model__learning_rate': 0.13766134492174428, ..."
1,gradient_boosting,2,0.776646,0.023088,0.835714,0.818258,0.743502,"{'model__learning_rate': 0.048917628051312216,..."
2,gradient_boosting,3,0.770301,0.022923,0.830119,0.806942,0.739798,"{'model__learning_rate': 0.08167611317882542, ..."
3,gradient_boosting,4,0.768194,0.015086,0.828671,0.804106,0.740000,"{'model__learning_rate': 0.018861950443028862,..."
4,gradient_boosting,5,0.767458,0.019255,0.830090,0.814485,0.728889,"{'model__learning_rate': 0.012184186502221764,..."
5,gradient_boosting,6,0.765190,0.025578,0.828691,0.813228,0.725185,"{'model__learning_rate': 0.019950144748611585,..."
6,gradient_boosting,7,0.764931,0.033651,0.820260,0.774016,0.758316,"{'model__learning_rate': 0.08360776902859286, ..."
7,gradient_boosting,8,0.764054,0.034408,0.827283,0.804965,0.728889,"{'model__learning_rate': 0.010461499920091565,..."
8,gradient_boosting,9,0.763858,0.015958,0.825864,0.800591,0.732593,"{'model__learning_rate': 0.028797752657070342,..."
9,gradient_boosting,10,0.763468,0.022671,0.827292,0.809856,0.725185,"{'model__learning_rate': 0.049833191601257244,..."


In [14]:
display(
    random_trials["adaboost"].head(10)
)

,model,rank_test_f1,mean_test_f1,std_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,params
0,adaboost,1,0.775455,0.025421,0.827253,0.775955,0.776566,"{'model__learning_rate': 0.46225890010208287, ..."
1,adaboost,2,0.772478,0.024511,0.824436,0.769767,0.776566,"{'model__learning_rate': 0.5246634533625282, '..."
2,adaboost,3,0.769285,0.031192,0.820240,0.759566,0.780269,"{'model__learning_rate': 0.9256818992066873, '..."
3,adaboost,4,0.767436,0.035673,0.818842,0.760329,0.776566,"{'model__learning_rate': 0.8536189862866826, '..."
4,adaboost,5,0.764742,0.035171,0.824456,0.784423,0.747205,"{'model__learning_rate': 0.2607024758370766, '..."
5,adaboost,6,0.750608,0.040247,0.814656,0.777834,0.728889,"{'model__learning_rate': 0.05404103854647329, ..."
6,adaboost,6,0.750608,0.040247,0.814656,0.777834,0.728889,"{'model__learning_rate': 0.06305535040199284, ..."
7,adaboost,8,0.748930,0.039959,0.814666,0.783862,0.721481,"{'model__learning_rate': 0.21137059440645725, ..."
8,adaboost,9,0.748424,0.034096,0.814626,0.779491,0.721616,"{'model__learning_rate': 0.15751320499779725, ..."
9,adaboost,10,0.747041,0.033125,0.813228,0.776796,0.721616,"{'model__learning_rate': 0.05899741796710491, ..."


In [15]:
display(
    random_trials["lr"].head(10)
)

,model,rank_test_f1,mean_test_f1,std_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,params
0,lr,1,0.773422,0.027021,0.828661,0.792473,0.758316,"{'model__C': 2.6373339933815254, 'model__class..."
1,lr,2,0.771174,0.020965,0.828681,0.794919,0.751044,"{'model__C': 1.0907475835157696, 'model__class..."
2,lr,3,0.770857,0.026978,0.827273,0.790686,0.754680,"{'model__C': 4.5705630998014515, 'model__class..."
3,lr,4,0.770367,0.035563,0.816054,0.743210,0.802290,"{'model__C': 0.5414413211338525, 'model__class..."
4,lr,5,0.770154,0.037287,0.816045,0.746970,0.798653,"{'model__C': 1.1462107403425035, 'model__class..."
5,lr,6,0.769299,0.037374,0.814636,0.741858,0.802290,"{'model__C': 0.9761125443110458, 'model__class..."
6,lr,6,0.769299,0.037374,0.814636,0.741858,0.802290,"{'model__C': 1.0129197956845732, 'model__class..."
7,lr,8,0.768813,0.018549,0.827273,0.794239,0.747340,"{'model__C': 0.9163741808778786, 'model__class..."
8,lr,9,0.768054,0.032040,0.827263,0.796384,0.743636,"{'model__C': 0.19069966103000435, 'model__clas..."
9,lr,10,0.766684,0.042702,0.811839,0.741312,0.798653,"{'model__C': 67.32248920775338, 'model__class_..."


In [16]:
display(
    random_trials["knn"].head(10)
)

,model,rank_test_f1,mean_test_f1,std_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,params
0,knn,1,0.747685,0.029837,0.820240,0.811999,0.695758,"{'model__metric': 'manhattan', 'model__n_neigh..."
1,knn,2,0.747208,0.039349,0.813247,0.775393,0.721549,"{'model__metric': 'manhattan', 'model__n_neigh..."
2,knn,3,0.744814,0.028287,0.806185,0.754524,0.736229,"{'model__metric': 'euclidean', 'model__n_neigh..."
3,knn,4,0.744738,0.030902,0.813228,0.785804,0.710438,"{'model__metric': 'euclidean', 'model__n_neigh..."
4,knn,5,0.743487,0.026080,0.813198,0.787160,0.706734,"{'model__metric': 'euclidean', 'model__n_neigh..."
5,knn,6,0.742433,0.026717,0.824416,0.853733,0.659327,"{'model__metric': 'manhattan', 'model__n_neigh..."
6,knn,7,0.741500,0.015283,0.806205,0.759444,0.725522,"{'model__metric': 'manhattan', 'model__n_neigh..."
7,knn,8,0.741181,0.030130,0.809022,0.770991,0.714343,"{'model__metric': 'manhattan', 'model__n_neigh..."
8,knn,9,0.739277,0.024186,0.818832,0.829653,0.670168,"{'model__metric': 'manhattan', 'model__n_neigh..."
9,knn,10,0.737748,0.029980,0.816035,0.817494,0.673939,"{'model__metric': 'manhattan', 'model__n_neigh..."


In [27]:
TOP_N = 10
TARGET_MODELS = ["lr", "rf", "gradient_boosting"]

for model_name in TARGET_MODELS:

    top_trials = random_trials[model_name].head(TOP_N).copy()

    # params 딕셔너리 펼치기
    params_df = pd.json_normalize(top_trials["params"])

    # model__ 접두사 제거
    params_df.columns = (
        params_df.columns
        .str.replace("model__", "", regex=False)
    )

    print(f"\n===== {model_name.upper()} TOP {TOP_N} PARAM SUMMARY =====")

    # 숫자형 파라미터 요약
    numeric_params = params_df.select_dtypes(include="number")

    if not numeric_params.empty:
        numeric_summary = (
            numeric_params
            .describe()
            .T[["min", "mean", "50%", "max"]]
            .rename(columns={"50%": "median"})
            .round(4)
        )

        display(numeric_summary)
    try:
        # 범주형 파라미터 요약
        categorical_params = params_df.select_dtypes(exclude="number")
        assert len(categorical_params.columns) != 0
        categorical_summary = []

        for col in categorical_params.columns:

            counts = categorical_params[col].value_counts(dropna=False)

            for value, count in counts.items():
                categorical_summary.append({
                    "parameter": col,
                    "value": value,
                    "count": count,
                    "ratio": count / TOP_N,
                })

        categorical_summary = pd.DataFrame(categorical_summary)

        display(categorical_summary)
    except:
        print('범주형 변수가 없습니다.')


===== LR TOP 10 PARAM SUMMARY =====


,min,mean,median,max
C,0.1907,8.0405,1.0518,67.3225


,parameter,value,count,ratio
0,class_weight,NaN,5,0.5
1,class_weight,balanced,5,0.5
2,penalty,l1,6,0.6
3,penalty,l2,4,0.4
4,solver,liblinear,10,1.0



===== RF TOP 10 PARAM SUMMARY =====


,min,mean,median,max
max_depth,5.0,15.5,15.0,30.0
min_samples_leaf,1.0,3.2,3.0,8.0
min_samples_split,5.0,9.8,10.5,13.0
n_estimators,340.0,601.1,617.5,791.0


,parameter,value,count,ratio
0,class_weight,balanced,9,0.9
1,class_weight,NaN,1,0.1
2,criterion,entropy,6,0.6
3,criterion,gini,4,0.4
4,max_features,log2,5,0.5
5,max_features,sqrt,4,0.4
6,max_features,NaN,1,0.1



===== GRADIENT_BOOSTING TOP 10 PARAM SUMMARY =====


,min,mean,median,max
learning_rate,0.0105,0.0492,0.0389,0.1377
max_depth,1.0000,3.5000,4.0000,5.0000
min_samples_leaf,2.0000,5.2000,4.5000,10.0000
min_samples_split,2.0000,10.6000,9.5000,20.0000
n_estimators,54.0000,214.8000,220.0000,395.0000
subsample,0.5488,0.7422,0.7529,0.9436


범주형 변수가 없습니다.


In [ ]:

# param_grids = {
#     "lr": {
#         "model__C": [0.01, 0.1, 1, 10, 30, 50, 70],
#         "model__penalty": ["l1", "l2"],
#         "model__solver": ["liblinear"],
#         "model__class_weight": ['balanced', None]
#     },

#     "rf": {
#         "model__n_estimators": [300, 400, 500, 600, 700, 800],
#         "model__max_depth": [5, 10, 15, 20, 30],
#         "model__min_samples_split": [2, 5, 10, 15],
#         "model__min_samples_leaf": [1, 2, 4, 8],
#         "model__max_features": ["sqrt"],
#     },

#     "gradient_boosting": {
#         "model__n_estimators": [50, 100, 200, 300, 400],
#         "model__learning_rate": [0.05, 0.1, 0.2],
#         "model__max_depth": [1, 2, 3, 4, 5],
#         "model__min_samples_leaf": [2, 3, 4, 5, 7, 10],
#         "model__min_samples_split": [2, 3, 4, 5, 10, 20],
#         "model__subsample": [0.7, 1.0],
#     },
# }


param_grids = {
    "lr": {
        "model__C": [2, 5, 7, 10, 15, 20],
        "model__penalty": ["l2"],
        "model__solver": ["liblinear"],
        "model__class_weight": [None, "balanced"],
    },

    "rf": {
        "model__n_estimators": [600, 700, 800],
        "model__max_depth": [10, 15, 20],
        "model__min_samples_split": [2, 3, 5],
        "model__min_samples_leaf": [1, 2, 3],
        "model__max_features": ["sqrt"],
    },

    "gradient_boosting": {
        "model__n_estimators": [75, 100, 125, 150],
        "model__learning_rate": [0.075, 0.1, 0.125],
        "model__max_depth": [2, 3, 4],
        "model__min_samples_leaf": [2, 3, 4],
        "model__min_samples_split": [15, 20, 25, 30],
        "model__subsample": [0.7, 0.8, 0.9],
    },
}

In [39]:

selected_model_configs = {
    name: model_configs[name]
    for name in TARGET_MODELS
}

grid_results, best_models = run_grid_search(
    model_configs=selected_model_configs,
    param_grids=param_grids,
    X_train=X_train,
    y_train=y_train,
    scoring=scoring,
    cv=cv
)

result_df = pd.DataFrame(grid_results).T

display(
    result_df.sort_values(
        "cv_f1",
        ascending=False
    )
)

/Users/yundong-ug/dev/boot_camp/kaggle-titanic-ml/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/yundong-ug/dev/boot_camp/kaggle-titanic-ml/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty

,best_params,cv_accuracy,cv_precision,cv_recall,cv_f1
gradient_boosting,"{'model__learning_rate': 0.1, 'model__max_dept...",0.841318,0.820339,0.758249,0.786079
lr,"{'model__C': 5, 'model__class_weight': None, '...",0.831478,0.794049,0.765657,0.777882
rf,"{'model__max_depth': 10, 'model__max_features'...",0.835753,0.820668,0.736229,0.775513


In [40]:
best_gb = best_models["gradient_boosting"]

y_pred = best_gb.predict(X_val)
y_proba = best_gb.predict_proba(X_val)[:, 1]

print("accuracy :", accuracy_score(y_val, y_pred))
print("precision:", precision_score(y_val, y_pred))
print("recall   :", recall_score(y_val, y_pred))
print("f1       :", f1_score(y_val, y_pred))
print("y_proba  :", y_proba)

accuracy : 0.8268156424581006
precision: 0.796875
recall   : 0.7391304347826086
f1       : 0.7669172932330827
y_proba  : [0.07490307 0.06354784 0.14567767 0.06501644 0.5989902  0.53619528
 0.83678667 0.88751156 0.50212452 0.1064153  0.13201385 0.14104844
 0.59073988 0.12494682 0.1202399  0.2391503  0.03006075 0.09089774
 0.02508779 0.86923999 0.09089774 0.81697272 0.11328924 0.51691556
 0.09101937 0.96812364 0.22642435 0.25354981 0.1123017  0.05697254
 0.15713704 0.93010078 0.87286551 0.13249739 0.30750758 0.79178176
 0.31582403 0.1095108  0.13201385 0.90255593 0.55168537 0.9566209
 0.09156451 0.80271859 0.84745048 0.18734836 0.127063   0.51643066
 0.93575974 0.86637798 0.9710633  0.27780454 0.79778287 0.08222731
 0.46702177 0.13149498 0.07583801 0.8121439  0.57500197 0.47565392
 0.30065836 0.89693582 0.17042367 0.05332753 0.83230257 0.45817443
 0.96960234 0.20434566 0.05570253 0.07068812 0.90536276 0.07933597
 0.89677095 0.97053829 0.14428933 0.05456884 0.1506744  0.11441666
 0.106110